# Error Handling, Debugging & Assertions (5+ Years Interview Guide)
Exhaustive revision guide to try/except/else/finally, exception chaining (raise from), custom hierarchies, and defensive assertions on transaction data.

### Key 5-Year Interview Concepts Covered:
- **Exception Interception**: Dedicated cell for `try`, `except (Err1, Err2) as e`, `else`, and `finally`.
- **Raising & Exception Chaining**: Dedicated cell for `raise` and `raise NewErr() from original_err`.
- **Custom Exception Hierarchies**: Dedicated cell for user-defined exception classes inheriting from `Exception`.
- **Defensive Assertions**: Dedicated cell for `assert condition, 'Error message'`.

This interactive revision guide uses `data/raw_transactions.csv` with individual dedicated cells per method.

In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from data/raw_transactions.csv


### Exception Interception: `try / except / else / finally`
**Explanation**: `else:` executes only if no exception occurred; `finally:` ALWAYS executes (even after `return` or unhandled exceptions) for critical cleanup.

**Syntax**: `try: ... except Err as e: ... else: ... finally: ...`

In [2]:
def parse_amount(val_str):
    try:
        amt = float(val_str)
    except (ValueError, TypeError) as err:
        print(f'Parsing Error intercepted: {err}')
        return 0.0
    else:
        return amt
    finally:
        pass # Cleanup hook

print('Valid parsing:', parse_amount(transactions[0]['transaction_amount']))
print('Corrupt parsing:', parse_amount('INVALID_NUMBER'))

Valid parsing: 1216.33
Parsing Error intercepted: could not convert string to float: 'INVALID_NUMBER'
Corrupt parsing: 0.0


### Custom Exception Hierarchies
**Explanation**: Defines application-specific exception domain models inheriting from base `Exception`.

**Syntax**: `class FraudDetectionError(Exception): pass`

In [3]:
class FintechError(Exception):
    """Base exception for all fintech banking errors."""
    pass

class SuspiciousTransactionError(FintechError):
    def __init__(self, tx_id, amount):
        super().__init__(f'Transaction {tx_id} exceeds velocity limit with amount ${amount}')
        self.tx_id = tx_id
        self.amount = amount

print('Custom exception class hierarchy created successfully.')

Custom exception class hierarchy created successfully.


### Exception Chaining with `raise ... from ...`
**Explanation**: Chains root cause exceptions using PEP 3134 (`raise NewError() from orig_error`) preserving original traceback context.

**Syntax**: `raise CustomError() from err`

In [4]:
try:
    try:
        float('CORRUPT_DATA')
    except ValueError as root_err:
        raise SuspiciousTransactionError('TX_UNKNOWN', 0.0) from root_err
except SuspiciousTransactionError as caught:
    print(f'Caught chained exception: {caught}')
    print(f'Underlying root cause: {caught.__cause__}')

Caught chained exception: Transaction TX_UNKNOWN exceeds velocity limit with amount $0.0
Underlying root cause: could not convert string to float: 'CORRUPT_DATA'


### Defensive Debugging with `assert`
**Explanation**: `assert condition, message` validates internal invariants during development. (Disabled in production when run with `python -O`).

**Syntax**: `assert amount >= 0.0, 'Negative amount detected'`

In [5]:
amt = float(transactions[0]['transaction_amount'])
assert amt >= 0.0, f'Transaction amount cannot be negative! Got: {amt}'
print(f'Assertion passed for amount ${amt:.2f}')

Assertion passed for amount $1216.33


## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Bare `except:` vs `except Exception:`
**Explanation**: Explain why bare `except:` is an anti-pattern: it catches `SystemExit`, `KeyboardInterrupt`, and `GeneratorExit`, preventing graceful process termination.

**Syntax**: `except Exception:` vs `except:`

In [6]:
print('Always catch specific exceptions or `except Exception:` to allow KeyboardInterrupt to propagate.')

Always catch specific exceptions or `except Exception:` to allow KeyboardInterrupt to propagate.
